### TODO

Read [TicTacToe - AlphaZero](https://github.com/Arnav235/ultimate_tic-tac-toe_alphazero/blob/master/comp_vs_human.py)

In [1]:
import time
import sys
from gomoku import Gomoku
from mcts import Tree, Node
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output

In [2]:
TIMEOUT = 0.5

In [3]:
import threading

class TimeoutError(Exception):
    pass

def timeout(seconds):
    def decorator(function):
        def wrapper(*args, **kwargs):
            stop_thread = threading.Event()
            def function_with_stop_check(*args, **kwargs):
                while not stop_thread.is_set():
                    function(*args, **kwargs)
            thread = threading.Thread(target=function_with_stop_check, args=args, kwargs=kwargs)
            thread.start()
            thread.join(timeout=seconds)
            if thread.is_alive():
                stop_thread.set()
                raise TimeoutError
        return wrapper
    return decorator

In [4]:
class Player:
    def next_move(self, game: Gomoku) -> tuple[int, int]:
        raise NotImplementedError

class RandomPlayer(Player):
    def next_move(self, game: Gomoku) -> tuple[int, int]:
        moves = game.actions()
        random_move  = np.random.randint(0, len(moves))
        return moves[random_move]
    
# class FlatPlayer(Player):
#     def __init__(self, max_moves: int = 0):
#         self.max_moves = max_moves
    
#     def next_move(self, game: Gomoku):
#         moves = game.actions()
#         if self.max_moves:
#             moves = moves[:self.max_moves]
        
#         max_reward = -np.inf
#         argmax_reward = None
        
#         for move in moves:
#             new_game = game.copy()
#             reward, _ = new_game.play(move)
#             if reward > max_reward:
#                 max_reward = reward
#                 argmax_reward = move
        
#         if argmax_reward is not None:
#             move = argmax_reward
#         else:
#             move = moves[np.random.randint(0, len(moves))]
        
#         return move
            
class MCTSPlayer(Player):
    def __init__(self, iterations=1000):
        self.iterations = iterations

    def next_move(self, game: Gomoku):
        tree = Tree(game)
        
        @timeout(TIMEOUT)
        def iterate():
            for _ in range(self.iterations):
                node = tree.select()
                value = tree.simulate(node)
                tree.backpropagate(node, value)
                
        try:
            iterate()
        except TimeoutError as e:
            print(e)
        
        best_child = max(tree.root.children, key=lambda child: child.Q)
        return best_child.state.history[-1]
    
class MCTSQPlayer(Player):
    def __init__(self, iterations=1000, max_depth=10, model=None):
        self.iterations = iterations
        self.max_depth = max_depth
        self.model = model
    
    def simulate(self, node: Node):
        state = node.state.copy(include_history=True)
        for _ in range(self.max_depth):
            if not state.fin():
                state_actions = state.actions()
                if not len(state_actions):
                    break
                action = state_actions[np.random.randint(0, len(state_actions))]
                state.play(action)
            else:
                if self.model is not None:
                    return self.model(state)
                else:
                    return np.random.random()
        return state.score()
        
    def next_move(self, game: Gomoku):
        tree = Tree(game)
        
        @timeout(TIMEOUT)
        def iterate():
            for _ in range(self.iterations):
                node = tree.select()
                value = tree.simulate(node)
                tree.backpropagate(node, value)
        
        try:
            iterate()
        except TimeoutError as e:
            print(e)
        
        best_child = max(tree.root.children, key=lambda child: child.Q)
        return best_child.state.history[-1]

In [8]:
def my_print(*args, **kwargs):
    clear_output(wait=True)
    print(*args, **kwargs)

def play_with_bot(game: Gomoku, opponent: Player):
    while not game.fin():
        sys.stdout.flush()
        if game.player == 1:
            move = opponent.next_move(game)
        else:
            moveStr = input("Enter move: ")
            move = tuple(map(int, [x for x in map(lambda x: x.strip(), moveStr.split(" ")) if x != ""]))
        game.play(move)
        
        game.print(my_print)
    return game.history, game.winner

def play_game(game: Gomoku, player1: Player, player2: Player, print = None):
    while not game.fin():
        if game.player == 1:
            move = player1.next_move(game)
        else:
            move = player2.next_move(game)
        game.play(move)
        
    if print is not None:
        game.print()
    return game.winner

def play_n_games(game: Gomoku, player1: Player, player2: Player, n: int, print = None):
    avg = 0
    avgs = []
    for i in range(n):
        score = (play_game(game.copy(), player1, player2, print) + 1) / 2
        avg *= i / (i + 1)
        avg += score / (i + 1)
        avgs += [avg]
    return avgs

def time_series_plot(ax, arr: list[float], title: str = ""):
    ax.plot(arr)
    ax.set_title(title)
    ax.set_xlabel("Game")
    ax.set_ylabel("Average score")

In [9]:
players = {
    "random": RandomPlayer(), # plays randomly
    # "flat": FlatPlayer(), # checks if any move is winning
    "mcts": MCTSPlayer(), # uses MCTS to evaluate the best move
    "mctsq": MCTSQPlayer(), # uses MCTS to evaluate the best move until a certain depth
}

In [47]:
# n = 25
# scores = {}
# for k in tqdm(range(len(players) ** 2)):
#     player_list, n_players = list(players), len(players)
#     i, j = k // n_players, k % n_players
#     playerI, playerJ = player_list[i], player_list[j]
#     game = Gomoku(
#         M=5,
#         N=5,
#         K=3,
#         FIRST_PLAYER=1,
#     )
#     score = play_n_games(game, players[playerI], players[playerJ], n, print=False)
#     scores[(playerI, playerJ)] = score
#
# fig, axes = plt.subplots(len(players), len(players), figsize=(20, 10))
# player_nums = dict([(v, i) for i, v in enumerate(players)])
# for k, v in scores.items():
#     playerI, playerJ = k
#     ax = axes[player_nums[playerI],  player_nums[playerJ]]
#     time_series_plot(ax, v, f"{playerI} vs {playerJ}")
# plt.show()

43


In [10]:
play_with_bot(game=Gomoku(M=6, N=6, K=4, FIRST_PLAYER=1), opponent=players["mcts"])

Current player: -1
. . . . . .
. . . . . .
. . O X . .
. . . X . .
. . . . . .
. . . . . .



ValueError: not enough values to unpack (expected 2, got 0)